# Lab 4 — Lakehouse Maintenance & Compaction

Your lakehouse works. Now let's keep it working. Left alone, every Iceberg table slowly degrades:

- **The small files problem** — streaming jobs, micro-batches, and over-partitioning produce thousands of tiny Parquet files. Query engines pay a fixed cost *per file* (S3 request, open, footer read, task scheduling), so 600 × 50 KB files can be dramatically slower to scan than 4 × 8 MB files holding the same rows.
- **Snapshot pile-up** — every write keeps the old snapshots (that's what made Lab 3's time travel possible). The metadata grows, and *no old data file can ever be deleted* while a snapshot still references it.
- **Orphan files** — failed or killed jobs leave behind data files that no snapshot references at all.

In this final lab you will **create the disease, then administer the cure**:

1. Simulate months of bad ingestion → **hundreds of tiny files** (in one fast write, no loops).
2. **Audit** the damage with the `.files` metadata table.
3. **Compact** with `rewrite_data_files`.
4. **Expire snapshots** to make the old small files deletable.
5. **Remove orphan files** and do a final physical audit in MinIO.

> **Prerequisite:** the Lab 1 stack is running (see `README.md`). This lab is self-contained — it creates its own table.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("Lab04-Maintenance").getOrCreate()
print(f"Spark {spark.version} — default catalog: {spark.conf.get('spark.sql.defaultCatalog')}")

## Step 1 — Simulate the small files problem

We generate **50,000 synthetic clickstream rows** spanning 4 days, in a single vectorized operation (`spark.range` — never Python loops or row-by-row inserts for this).

The table is sensibly partitioned by `days(event_ts)` — the partitioning is *not* the villain here. The villain is **how we write**: `repartition(150)` splits the data into 150 random shuffle chunks, so every task holds rows for *all 4 days* and writes its own tiny file into *each* partition — exactly the file pattern you get from months of small micro-batch ingestion, compressed into one command.

**One guardrail to disarm first:** modern Iceberg (1.2+) defends against sloppy writes with `write.distribution-mode = hash` by default — it re-shuffles incoming data by partition key so each partition is written by one task. That prevents small files, though it isn't free: on very large writes the extra shuffle costs network time and can skew/OOM if partitions are unbalanced, which is why teams tune it. We set it to `none` to behave like the many legacy/streaming pipelines that write unshuffled.

> ⚠️ **The even worse anti-pattern (don't do this for real):** partitioning by a high-cardinality column — a UUID, a raw timestamp, a session ID. That guarantees one micro-file *per value*, and — think about it — **compaction can't even fix it**, because `rewrite_data_files` only merges files *within* a partition. The only cure there is changing the partition spec (partition evolution). Small files you can fix; bad partition specs you have to *migrate*.

In [ ]:
# 50k synthetic clickstream events across 4 days (2026-07-01 .. 2026-07-04)
df_clicks = (
    spark.range(50_000).withColumnRenamed("id", "event_id")
    .withColumn("user_id",     (F.col("event_id") % 5000).cast("int"))
    .withColumn("page_url",    F.concat(F.lit("/product/"), (F.col("event_id") % 300).cast("string")))
    .withColumn("event_ts",    F.expr("timestamp'2026-07-01 00:00:00' + make_interval(0,0,0,0,0,0, (event_id * 7) % 345600)"))
    .withColumn("duration_ms", (F.rand(seed=42) * 60000).cast("int"))
)
print(f"Generated {df_clicks.count()} rows, "
      f"{df_clicks.select(F.to_date('event_ts')).distinct().count()} distinct days")

spark.sql("DROP TABLE IF EXISTS hive_prod.db.bloated_table PURGE")
spark.sql("""
    CREATE TABLE hive_prod.db.bloated_table (
        event_id    BIGINT,
        user_id     INT,
        page_url    STRING,
        event_ts    TIMESTAMP,
        duration_ms INT
    )
    USING iceberg
    PARTITIONED BY (days(event_ts))
    TBLPROPERTIES (
        -- disarm Iceberg's anti-small-files guardrail to mimic legacy pipelines
        'write.distribution-mode' = 'none'
    )
""")

# The bad write: 150 shuffle chunks x 4 day-partitions = ~600 tiny files, one commit
(df_clicks
    .repartition(150)
    .sortWithinPartitions("event_ts")   # each task groups its rows by partition before writing
    .writeTo("hive_prod.db.bloated_table")
    .append())

print("Write committed.")

## Step 2 — Audit the damage with the `.files` metadata table

No filesystem crawling needed: Iceberg's manifests already know every data file and its size. First the raw view, then the summary that a platform engineer would put on a dashboard:

In [ ]:
spark.sql("""
    SELECT file_path, file_size_in_bytes
    FROM hive_prod.db.bloated_table.files
""").show(5, truncate=80)

audit = spark.sql("""
    SELECT count(*)                            AS file_count,
           round(avg(file_size_in_bytes)/1024, 1) AS avg_kb,
           round(min(file_size_in_bytes)/1024, 1) AS min_kb,
           round(max(file_size_in_bytes)/1024, 1) AS max_kb,
           round(sum(file_size_in_bytes)/1024/1024, 2) AS total_mb
    FROM hive_prod.db.bloated_table.files
""")
audit.show()

baseline_file_count = audit.collect()[0].file_count
print(f"BASELINE: {baseline_file_count} data files before compaction")

Hundreds of files averaging a few dozen **kilo**bytes — for ~2 MB of actual data per day. Every query against this table now pays ~600 S3 GETs just to start scanning.

## Step 3 — Compaction: `rewrite_data_files`

Iceberg ships maintenance as **stored procedures** you `CALL` from SQL. `rewrite_data_files` bin-packs small files into optimally-sized ones (512 MB target by default), **within each partition**. Crucially, this is an *online* operation: it writes the merged files first, then commits a new snapshot — concurrent readers keep reading the old snapshot until the atomic swap.

In [ ]:
result = spark.sql("""
    CALL hive_prod.system.rewrite_data_files(table => 'db.bloated_table')
""")
result.show()

# Useful options for real deployments (try them later):
#   options => map('target-file-size-bytes','134217728')   -- e.g. 128 MB target
#   strategy => 'sort', sort_order => 'user_id'             -- cluster data while compacting
#   where => 'event_ts >= TIMESTAMP \'2026-07-04 00:00:00\''  -- compact only hot partitions

## Step 4 — Verify: re-run the audit

In [ ]:
audit_after = spark.sql("""
    SELECT count(*)                            AS file_count,
           round(avg(file_size_in_bytes)/1024, 1) AS avg_kb,
           round(sum(file_size_in_bytes)/1024/1024, 2) AS total_mb
    FROM hive_prod.db.bloated_table.files
""")
audit_after.show()

compacted_file_count = audit_after.collect()[0].file_count
print(f"{baseline_file_count} files  →  {compacted_file_count} files "
      f"({baseline_file_count // max(compacted_file_count,1)}x reduction)")

# Same rows, same result — compaction is invisible to queries:
spark.sql("SELECT count(*) AS row_count FROM hive_prod.db.bloated_table").show()

~600 files became **one file per day-partition**. 

### 🧠 Key conceptual note — the small files are still there!

Open MinIO (<http://localhost:9001>, `admin`/`password`) and browse `warehouse/db.db/bloated_table/data/event_ts_day=2026-07-01/`. The hundreds of tiny Parquet files **still physically exist**. Why? Because the pre-compaction **snapshot still references them** — if Iceberg deleted them, Lab 3's time travel would break. Compaction made the *current* snapshot fast; it cannot touch history.

The metadata tables show both worlds at once — files in the **current** snapshot vs files across **all** snapshots:

In [ ]:
spark.sql("""
    SELECT 'current snapshot (.files)' AS view, count(*) AS data_files
    FROM hive_prod.db.bloated_table.files
    UNION ALL
    SELECT 'all snapshots (.all_files)', count(*)
    FROM hive_prod.db.bloated_table.all_files
""").show(truncate=False)

spark.sql("""
    SELECT snapshot_id, committed_at, operation, summary['total-data-files'] AS total_data_files
    FROM hive_prod.db.bloated_table.snapshots ORDER BY committed_at
""").show(truncate=False)

## Step 5 — Expire snapshots to release the old files

`expire_snapshots` removes old snapshots from metadata **and physically deletes** every data file that only they referenced. This is the moment the storage is actually reclaimed — and the moment you give up time travel beyond the retention window. Production wisdom: run it on a schedule with a business-approved retention (e.g. keep 7 days), never ad-hoc.

Here we keep only the latest snapshot (`retain_last => 1`, `older_than =>` right now) — maximum cleanup, zero history:

In [ ]:
from datetime import datetime, timedelta, timezone

cutoff = (datetime.now(timezone.utc) + timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S")

spark.sql(f"""
    CALL hive_prod.system.expire_snapshots(
        table       => 'db.bloated_table',
        older_than  => TIMESTAMP '{cutoff}',
        retain_last => 1
    )
""").show()

In [ ]:
# The audit trail is gone (1 snapshot), and .all_files == .files:
spark.sql("SELECT count(*) AS snapshots FROM hive_prod.db.bloated_table.snapshots").show()
spark.sql("""
    SELECT 'current snapshot (.files)' AS view, count(*) AS data_files
    FROM hive_prod.db.bloated_table.files
    UNION ALL
    SELECT 'all snapshots (.all_files)', count(*)
    FROM hive_prod.db.bloated_table.all_files
""").show(truncate=False)

`deleted_data_files_count` in the procedure output ≈ your baseline file count: the tiny files were **physically deleted from MinIO** during expiry.

## Step 6 — Remove orphan files

Expiry handles files that *snapshots* referenced. **Orphans** are different: files sitting inside the table's directory that **no snapshot has ever referenced** — debris from failed jobs, killed containers, aborted commits. Iceberg's metadata can't see them (that's the point), so `remove_orphan_files` compares the *actual* bucket listing against *every* file referenced in metadata and deletes the difference.

Let's manufacture some debris — a "failed job" that dumped Parquet into the table location without ever committing:

In [ ]:
# Find the table's physical location from the catalog
table_location = [
    r.data_type for r in
    spark.sql("DESCRIBE TABLE EXTENDED hive_prod.db.bloated_table").collect()
    if r.col_name == "Location"
][0]
print(f"Table location: {table_location}")

# A 'failed job' writes files into the table dir but never commits to the catalog.
# (No leading underscore in the folder name: '_'- and '.'-prefixed paths are
# treated as hidden and are invisible to orphan scans — that's also why the
# _SUCCESS marker files Spark drops everywhere never get swept.)
spark.range(1000).write.mode("overwrite").parquet(f"{table_location}/data/debris_from_failed_job")
print("Debris written. Iceberg metadata knows nothing about these files.")

# Proof: the table itself is unaffected
spark.sql("SELECT count(*) AS row_count FROM hive_prod.db.bloated_table").show()

In [ ]:
# First, the official procedure — asking it to sweep files younger than 24h:
try:
    spark.sql(f"""
        CALL hive_prod.system.remove_orphan_files(
            table      => 'db.bloated_table',
            older_than => TIMESTAMP '{(datetime.now(timezone.utc) + timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S")}'
        )
    """).show()
except Exception as e:
    if "interval less than 24 hours" not in str(e):
        raise   # only the expected safety refusal is tolerated here
    print("Iceberg REFUSED the sweep:\n")
    print(str(e).split("\n")[0])

**Iceberg said no — and that's the lesson.** A write job that is *currently running* has uncommitted files on disk that look exactly like orphans; an aggressive sweep would delete them mid-flight and corrupt the commit. So the SQL procedure hard-refuses any interval under **24 hours** (and defaults to 3 days). In production you accept that: run the procedure weekly with the default window, and today's debris gets swept next week.

In a lab we can't wait 24 hours. For exactly this situation Iceberg's error message points at the **Action API** — the lower-level Java interface the procedure wraps. We reach it through PySpark's JVM gateway (**lab-only escape hatch** — in production, use the procedure and its defaults):

In [ ]:
import time

SparkActions = spark._jvm.org.apache.iceberg.spark.actions.SparkActions
iceberg_table = spark._jvm.org.apache.iceberg.spark.Spark3Util.loadIcebergTable(
    spark._jsparkSession, "hive_prod.db.bloated_table")

result = (SparkActions.get(spark._jsparkSession)
          .deleteOrphanFiles(iceberg_table)
          .olderThan(int(time.time() * 1000) + 60_000)   # 'anything older than 1 min from now'
          .execute())

it = result.orphanFileLocations().iterator()
deleted = []
while it.hasNext():
    deleted.append(it.next())

print(f"Orphan files deleted: {len(deleted)}")
for path in deleted[:5]:
    print("  ", path)

Every deleted path is inside `data/debris_from_failed_job/` — the sweep found the debris by *listing the bucket* and subtracting everything Iceberg's metadata references. Note it deleted only the Parquet part-files: the `_SUCCESS` marker is `_`-prefixed (hidden) and invisible to the scan.

## Step 7 — Final MinIO audit 🔍

**Go look with your own eyes.** Open <http://localhost:9001> → Object Browser → `warehouse/db.db/bloated_table/`:

1. **`data/event_ts_day=2026-07-01/` … `=2026-07-04/`** — each day folder should now contain a **single large Parquet file**. The hundreds of tiny files are *physically gone*.
2. **`data/debris_from_failed_job/`** — the orphaned Parquet debris has been purged (at most the empty `_SUCCESS` marker remains).
3. **`metadata/`** — slimmed down too: expired snapshots' manifest files were also deleted.

Prefer the terminal? Count the physical data files from your host:

```bash
docker exec minio sh -c "mc alias set local http://localhost:9000 admin password >/dev/null && mc ls -r local/warehouse/db.db/bloated_table/data/ | wc -l"
```

You should see a number close to your **post-compaction** file count (4-ish, plus the hidden `_SUCCESS` marker) — not the ~600 you started with.

In [ ]:
# One last sanity check: after all that surgery, the data is intact.
spark.sql("""
    SELECT to_date(event_ts) AS day, count(*) AS events, sum(duration_ms) AS total_ms
    FROM hive_prod.db.bloated_table
    GROUP BY 1 ORDER BY 1
""").show()

## ✅ Wrap-up — and course conclusion 🎓

Today you:
- ✔ Reproduced the **small files problem** and measured it with metadata tables (no filesystem crawling)
- ✔ Ran **online compaction** (`rewrite_data_files`) — 600 → 4 files, readers never blocked
- ✔ Learned why compaction alone frees **zero bytes**, then reclaimed storage with `expire_snapshots`
- ✔ Swept genuinely unreferenced debris with `remove_orphan_files` — and learned why its `older_than` guard matters

**The production maintenance loop** (run scheduled, e.g. nightly):
1. `rewrite_data_files` — keep scans fast (optionally `strategy => 'sort'`)
2. `expire_snapshots` — enforce the retention window your business agreed to
3. `remove_orphan_files` — weekly, with a generous `older_than`
4. `rewrite_manifests` — bonus: compaction for the *metadata* layer (try it: `CALL hive_prod.system.rewrite_manifests(table => 'db.bloated_table')`)

**Across these four labs** you built a complete lakehouse (object storage + catalog + two engines), ingested and evolved schemas without downtime, time-traveled and rolled back production incidents, and operated the storage layer like an SRE. That *is* the Iceberg engineering job description. Congratulations! 🧊